# 🏎️ PPO Agent Training — Car Racing (Google Colab)

**Self-contained Colab notebook** for training a PPO (Proximal Policy Optimization) agent on OpenAI Gymnasium's `CarRacing-v3` environment.

### What this notebook does:
1. Mounts Google Drive for persistent checkpoint storage
2. Installs dependencies
3. Defines the full PPO pipeline (CNN policy, rollout buffer, GAE, reward shaping, preprocessing)
4. Trains the agent with periodic evaluation and checkpointing **to Google Drive**
5. Saves the **best model** as `best_model.pt` on Drive

### Resume after GPU disconnect:
- All checkpoints are saved to Google Drive every rollout
- If Colab disconnects, just **re-run all cells** — training resumes from the latest checkpoint automatically

### After training:
- Download `best_model.pt` from Google Drive (`Car_Racing/models/ppo_best/`)
- Place it at `models/ppo_best/best_model.pt` in your local project
- Run `python -m src.main race` to race against the AI

### Colab Settings:
- **Runtime → Change runtime type → GPU** (T4 recommended)
- **Internet**: ON (for pip install)

## 1. Install Dependencies

In [ ]:
!pip install -q gymnasium[box2d]>=0.29.0 torch>=2.0.0 numpy>=1.24.0 tqdm>=4.65.0 opencv-python>=4.8.0 matplotlib>=3.7.0

# Install torch_xla if running on TPU (pre-installed on Kaggle TPU runtimes)
import os
if "TPU_NAME" in os.environ or "COLAB_TPU_ADDR" in os.environ:
    print("TPU detected — installing torch_xla...")
    !pip install -q torch_xla
else:
    print("No TPU detected — will use GPU/CPU.")

## 1.5 Mount Google Drive (Persistent Storage)

Checkpoints are saved to Google Drive so they survive GPU disconnects.
If Colab restarts, just re-run all cells — training will resume from the latest checkpoint.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create checkpoint directories on Google Drive
DRIVE_BASE = "/content/drive/MyDrive/Car_Racing"
DRIVE_SAVE_DIR = os.path.join(DRIVE_BASE, "models", "ppo_baseline")
DRIVE_BEST_DIR = os.path.join(DRIVE_BASE, "models", "ppo_best")

os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
os.makedirs(DRIVE_BEST_DIR, exist_ok=True)

print(f"Google Drive mounted successfully!")
print(f"  Base directory:     {DRIVE_BASE}")
print(f"  Checkpoint dir:     {DRIVE_SAVE_DIR}")
print(f"  Best model dir:     {DRIVE_BEST_DIR}")

# Check if a previous checkpoint exists (for resume)
existing_checkpoints = sorted([
    f for f in os.listdir(DRIVE_SAVE_DIR) if f.startswith("checkpoint_") and f.endswith(".pt")
]) if os.path.exists(DRIVE_SAVE_DIR) else []

if existing_checkpoints:
    print(f"\n  Found {len(existing_checkpoints)} existing checkpoint(s) — training will RESUME from latest.")
    print(f"  Latest: {existing_checkpoints[-1]}")
else:
    print(f"\n  No existing checkpoints — training will start from scratch.")

## 2. Imports & Device Setup

In [ ]:
from __future__ import annotations

import collections
import logging
import math
import os
import pathlib
import random
import time
from typing import Any, SupportsFloat

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
from numpy.typing import NDArray
from torch import Tensor, nn
from tqdm.auto import tqdm, trange

try:
    import cv2 as _cv2
    _HAS_CV2 = True
except ImportError:
    _HAS_CV2 = False

# ---------- TPU support via torch_xla ----------
_HAS_XLA = False
try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    _HAS_XLA = True
except ImportError:
    pass

# ---------- Logging setup ----------
LOG_LEVEL = logging.DEBUG  # Change to logging.INFO to reduce verbosity

logger = logging.getLogger("ppo_training")
logger.setLevel(LOG_LEVEL)
logger.handlers.clear()

_console = logging.StreamHandler()
_console.setLevel(LOG_LEVEL)
_fmt = logging.Formatter(
    "[%(asctime)s] %(levelname)-8s %(message)s",
    datefmt="%H:%M:%S",
)
_console.setFormatter(_fmt)
logger.addHandler(_console)
logger.propagate = False

logger.info("Logger initialized at %s level", logging.getLevelName(LOG_LEVEL))

# ---------- Device auto-detection: TPU > GPU > CPU ----------
if _HAS_XLA:
    DEVICE = xm.xla_device()
    DEVICE_TYPE = "tpu"
    logger.info("Using device: TPU (%s)", DEVICE)
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    DEVICE_TYPE = "cuda"
    logger.info("Using device: %s", DEVICE)
    logger.info("GPU: %s", torch.cuda.get_device_name(0))
else:
    DEVICE = torch.device("cpu")
    DEVICE_TYPE = "cpu"
    logger.warning("Using device: CPU (training will be slow)")

logger.debug("PyTorch version: %s", torch.__version__)
logger.debug("NumPy version: %s", np.__version__)
logger.debug("Gymnasium version: %s", gym.__version__)
logger.debug("OpenCV available: %s", _HAS_CV2)
logger.debug("XLA available: %s", _HAS_XLA)

## 3. Reproducibility

In [ ]:
def set_global_seed(seed: int) -> None:
    """Set seeds for Python, NumPy, PyTorch, and CUDA/TPU for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        # NOTE: deterministic=False + benchmark=True for speed over exact reproducibility
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True
    if _HAS_XLA:
        import torch_xla.core.xla_model as xm
        xm.set_rng_state(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

SEED = 42
set_global_seed(SEED)
print(f"Global seed set to {SEED}")
print(f"  cudnn.benchmark = {torch.backends.cudnn.benchmark}  (faster convolutions)")

## 4. Configuration

All training hyperparameters and environment settings in one place.

In [ ]:
# =====================================================================
# ENVIRONMENT CONFIG
# =====================================================================
ENV_CONFIG = {
    "environment": {
        "name": "CarRacing-v3",
        "continuous": False,     # Discrete action space (5 actions)
        "max_episode_steps": 1000,
    },
    "preprocessing": {
        "grayscale": True,
        "resize": [84, 84],
        "frame_stack": 4,
        "frame_skip": 4,
        "normalize": True,
    },
    "reward_shaping": {
        "enabled": True,
        "speed_reward_weight": 0.1,
        "grass_penalty": -0.5,
        "backward_penalty": -1.0,
        "standing_still_penalty": -0.1,
        "tile_visit_bonus": 1.0,
    },
}

# =====================================================================
# TRAINING CONFIG  (speed-optimized — reduced from 1M to 500K timesteps)
# =====================================================================
# Speed changes:
#   - total_timesteps: 1M → 500K     (halves wall time: ~10h → ~5h)
#   - eval_freq:       100K → 250K   (only 2 evals instead of 10)
#   - n_eval_episodes: 3 → 2         (faster evaluations)
#   - save_freq:       200K → 500K   (save only at end + best)
#
# Stability fixes (kept from previous run):
#   - learning_rate:  2.5e-4, vf_coef: 0.25, ent_coef: 0.005
#   - clip_vf: True, reward_normalization: True
#
# Colab: checkpoints saved to Google Drive for persistence

TRAIN_CONFIG = {
    "training": {
        "total_timesteps": 500_000,   # HALVED: 1M → 500K (~5h instead of ~10h)
        "learning_rate": 2.5e-4,
        "n_steps": 1024,
        "batch_size": 128,
        "n_epochs": 3,
        "gamma": 0.99,
        "gae_lambda": 0.95,
        "clip_range": 0.2,
        "clip_vf": True,
        "clip_vf_range": 10.0,
        "ent_coef": 0.005,
        "vf_coef": 0.25,
        "max_grad_norm": 0.5,
        "reward_normalization": True,
        "seed": SEED,
        "device": "auto",
    },
    "checkpoints": {
        "save_dir": DRIVE_SAVE_DIR,
        "best_model_dir": DRIVE_BEST_DIR,
        "save_freq": 500_000,         # REDUCED: save only at end (was 200K)
    },
    "evaluation": {
        "eval_freq": 250_000,         # REDUCED: 2 evals total (was 100K → ~10 evals)
        "n_eval_episodes": 2,         # REDUCED: faster evals (was 3)
        "deterministic": True,
    },
}

N_ENVS = 8

print("Config loaded (speed-optimized, Colab + Google Drive).")
print(f"  Total timesteps: {TRAIN_CONFIG['training']['total_timesteps']:,}")
print(f"  Rollouts:        {TRAIN_CONFIG['training']['total_timesteps'] // (TRAIN_CONFIG['training']['n_steps'] * N_ENVS)}")
print(f"  Parallel envs:   {N_ENVS}")
print(f"  Batch size:      {TRAIN_CONFIG['training']['batch_size']}")
print(f"  n_steps:         {TRAIN_CONFIG['training']['n_steps']}")
print(f"  n_epochs:        {TRAIN_CONFIG['training']['n_epochs']}")
print(f"  Learning rate:   {TRAIN_CONFIG['training']['learning_rate']}")
print(f"  Eval freq:       {TRAIN_CONFIG['evaluation']['eval_freq']:,}  (was 100K)")
print(f"  Save freq:       {TRAIN_CONFIG['checkpoints']['save_freq']:,}  (was 200K)")
print(f"  Save dir:        {TRAIN_CONFIG['checkpoints']['save_dir']}")
print(f"  Best model dir:  {TRAIN_CONFIG['checkpoints']['best_model_dir']}")

## 5. Environment Wrappers

Frame preprocessing pipeline: Grayscale → Resize → Frame Skip → Frame Stack → Normalize

In [ ]:
# =====================================================================
# PREPROCESSING WRAPPERS
# =====================================================================

class GrayscaleWrapper(gym.ObservationWrapper):
    """Convert RGB observations to single-channel grayscale."""

    def __init__(self, env: gym.Env) -> None:
        super().__init__(env)
        h, w = self.observation_space.shape[:2]
        self.observation_space = gym.spaces.Box(0, 255, shape=(h, w, 1), dtype=np.uint8)

    def observation(self, obs: NDArray) -> NDArray:
        grey = np.dot(obs[..., :3], [0.2989, 0.5870, 0.1140]).astype(np.uint8)
        return grey[..., np.newaxis]


class ResizeWrapper(gym.ObservationWrapper):
    """Resize observations to target (H, W)."""

    def __init__(self, env: gym.Env, size: tuple[int, int]) -> None:
        super().__init__(env)
        self._size = size
        channels = self.observation_space.shape[2] if len(self.observation_space.shape) == 3 else 1
        self.observation_space = gym.spaces.Box(0, 255, shape=(*size, channels), dtype=np.uint8)

    def observation(self, obs: NDArray) -> NDArray:
        squeeze = obs.ndim == 3 and obs.shape[-1] == 1
        src = obs.squeeze(-1) if squeeze else obs
        if _HAS_CV2:
            resized = _cv2.resize(src, (self._size[1], self._size[0]), interpolation=_cv2.INTER_AREA)
        else:
            h, w = self._size
            oh, ow = src.shape[:2]
            row_idx = (np.arange(h) * oh // h).astype(int)
            col_idx = (np.arange(w) * ow // w).astype(int)
            resized = src[np.ix_(row_idx, col_idx)]
        resized = np.asarray(resized, dtype=np.uint8)
        if resized.ndim == 2:
            resized = resized[..., np.newaxis]
        return resized


class FrameSkipWrapper(gym.Wrapper):
    """Repeat each action for *skip* frames, summing rewards."""

    def __init__(self, env: gym.Env, skip: int = 4) -> None:
        super().__init__(env)
        self._skip = max(1, skip)

    def step(self, action):
        total_reward = 0.0
        for _ in range(self._skip):
            obs, reward, terminated, truncated, info = self.env.step(action)
            total_reward += float(reward)
            if terminated or truncated:
                break
        return obs, total_reward, terminated, truncated, info


class FrameStackWrapper(gym.Wrapper):
    """Stack the last *n* observations along the channel axis."""

    def __init__(self, env: gym.Env, n: int = 4) -> None:
        super().__init__(env)
        self._n = n
        low = np.repeat(env.observation_space.low, n, axis=-1)
        high = np.repeat(env.observation_space.high, n, axis=-1)
        self.observation_space = gym.spaces.Box(low=low, high=high, dtype=np.uint8)
        self._frames: collections.deque[NDArray] = collections.deque(maxlen=n)

    def reset(self, *, seed=None, options=None):
        obs, info = self.env.reset(seed=seed, options=options)
        for _ in range(self._n):
            self._frames.append(obs)
        return self._get_obs(), info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        self._frames.append(obs)
        return self._get_obs(), reward, terminated, truncated, info

    def _get_obs(self) -> NDArray:
        return np.concatenate(list(self._frames), axis=-1)


class NormalizeWrapper(gym.ObservationWrapper):
    """Scale pixel observations from [0, 255] to [0.0, 1.0]."""

    def __init__(self, env: gym.Env) -> None:
        super().__init__(env)
        self.observation_space = gym.spaces.Box(
            low=0.0, high=1.0, shape=self.observation_space.shape, dtype=np.float32
        )

    def observation(self, obs: NDArray) -> NDArray:
        return obs.astype(np.float32) / 255.0


class RewardNormWrapper(gym.Wrapper):
    """Normalize rewards using running mean/variance.
    
    This keeps rewards in a reasonable range (roughly [-5, 5]) so the value
    function can fit them without exploding.  Uses Welford's online algorithm.
    """

    def __init__(self, env: gym.Env, clip: float = 10.0, gamma: float = 0.99) -> None:
        super().__init__(env)
        self._clip = clip
        self._gamma = gamma
        # Running stats for discounted return (not raw reward) — recommended by PPO paper
        self._ret_mean = 0.0
        self._ret_var = 1.0
        self._ret_count = 1e-4
        self._discounted_ret = 0.0

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        info["raw_reward"] = float(reward)

        # Update running return
        self._discounted_ret = self._discounted_ret * self._gamma + float(reward)
        self._update_stats(self._discounted_ret)

        # Normalize
        reward = float(reward) / (math.sqrt(self._ret_var) + 1e-8)
        reward = max(min(reward, self._clip), -self._clip)

        if terminated or truncated:
            self._discounted_ret = 0.0

        return obs, reward, terminated, truncated, info

    def _update_stats(self, val: float) -> None:
        self._ret_count += 1
        delta = val - self._ret_mean
        self._ret_mean += delta / self._ret_count
        delta2 = val - self._ret_mean
        self._ret_var += (delta * delta2 - self._ret_var) / self._ret_count

    def reset(self, *, seed=None, options=None):
        self._discounted_ret = 0.0
        return self.env.reset(seed=seed, options=options)


print("Wrappers defined: GrayscaleWrapper, ResizeWrapper, FrameSkipWrapper, FrameStackWrapper, NormalizeWrapper, RewardNormWrapper")

## 6. Reward Shaping

In [ ]:
class RewardShaper(gym.Wrapper):
    """Augment the default reward signal with domain-specific shaping."""

    def __init__(
        self,
        env: gym.Env,
        *,
        speed_reward_weight: float = 0.1,
        grass_penalty: float = -0.5,
        backward_penalty: float = -1.0,
        standing_still_penalty: float = -0.1,
        tile_visit_bonus: float = 1.0,
    ) -> None:
        super().__init__(env)
        self._speed_w = speed_reward_weight
        self._grass_pen = grass_penalty
        self._backward_pen = backward_penalty
        self._still_pen = standing_still_penalty
        self._tile_bonus = tile_visit_bonus
        self._prev_tiles: int = 0
        self._step_count: int = 0
        self._car_env_cache: Any = None

    def reset(self, *, seed=None, options=None):
        obs, info = self.env.reset(seed=seed, options=options)
        self._prev_tiles = 0
        self._step_count = 0
        self._car_env_cache = self._unwrap_car_env()
        return obs, info

    def step(self, action):
        obs, base_reward, terminated, truncated, info = self.env.step(action)
        self._step_count += 1
        shaped_reward = float(base_reward)

        car_env = self._car_env_cache
        if car_env is not None:
            car = getattr(car_env, "car", None)
            if car is not None:
                speed = np.sqrt(car.hull.linearVelocity[0] ** 2 + car.hull.linearVelocity[1] ** 2)
                shaped_reward += self._speed_w * min(speed, 50.0) / 50.0

                if speed < 1.0 and self._step_count > 10:
                    shaped_reward += self._still_pen

                on_grass = any(
                    len(getattr(w, "tiles", {1})) == 0
                    for w in car.wheels
                )
                if on_grass:
                    shaped_reward += self._grass_pen

            tile_count = getattr(car_env, "tile_visited_count", 0)
            new_tiles = tile_count - self._prev_tiles
            if new_tiles > 0:
                shaped_reward += self._tile_bonus * new_tiles
                self._prev_tiles = tile_count

        info["shaped_reward"] = shaped_reward
        info["base_reward"] = float(base_reward)
        return obs, shaped_reward, terminated, truncated, info

    def _unwrap_car_env(self):
        env = self.env
        while hasattr(env, "env"):
            if type(env).__name__ == "CarRacing":
                return env
            env = env.env
        if type(env).__name__ == "CarRacing":
            return env
        return getattr(self, "unwrapped", None)


print("RewardShaper defined.")

## 7. Environment Factory

In [ ]:
def make_env(env_cfg: dict, seed: int = 0, render: bool = False, reward_norm: bool = True) -> gym.Env:
    """Build a fully-wrapped CarRacing environment."""
    e = env_cfg.get("environment", {})
    p = env_cfg.get("preprocessing", {})
    r = env_cfg.get("reward_shaping", {})

    render_mode = "human" if render else e.get("render_mode")
    logger.debug("Creating env '%s' (seed=%d, render=%s)", e.get("name", "CarRacing-v3"), seed, render_mode)

    env = gym.make(
        e.get("name", "CarRacing-v3"),
        continuous=e.get("continuous", False),
        render_mode=render_mode,
        max_episode_steps=e.get("max_episode_steps", 1000000),
    )

    env = FrameSkipWrapper(env, skip=p.get("frame_skip", 4))
    logger.debug("  + FrameSkipWrapper (skip=%d)", p.get("frame_skip", 4))

    if r.get("enabled", False):
        env = RewardShaper(
            env,
            speed_reward_weight=r.get("speed_reward_weight", 0.1),
            grass_penalty=r.get("grass_penalty", -0.5),
            backward_penalty=r.get("backward_penalty", -1.0),
            standing_still_penalty=r.get("standing_still_penalty", -0.1),
            tile_visit_bonus=r.get("tile_visit_bonus", 1.0),
        )
        logger.debug("  + RewardShaper (speed_w=%.2f, grass=%.2f)", r.get("speed_reward_weight", 0.1), r.get("grass_penalty", -0.5))

    # Reward normalization — keeps rewards in ~[-10, 10] so value function doesn't explode
    if reward_norm:
        env = RewardNormWrapper(env, clip=10.0, gamma=e.get("gamma", 0.99))
        logger.debug("  + RewardNormWrapper (clip=10.0)")

    if p.get("grayscale", True):
        env = GrayscaleWrapper(env)
        logger.debug("  + GrayscaleWrapper")

    resize = p.get("resize", [84, 84])
    env = ResizeWrapper(env, size=tuple(resize))
    logger.debug("  + ResizeWrapper (%s)", resize)

    env = FrameStackWrapper(env, n=p.get("frame_stack", 4))
    logger.debug("  + FrameStackWrapper (n=%d)", p.get("frame_stack", 4))

    if p.get("normalize", True):
        env = NormalizeWrapper(env)
        logger.debug("  + NormalizeWrapper")

    env.reset(seed=seed)
    logger.info("Env created: obs_shape=%s, action_space=%s", env.observation_space.shape, env.action_space)
    return env


def make_vec_env(env_cfg: dict, n_envs: int, seed: int = 0, reward_norm: bool = True) -> gym.vector.VectorEnv:
    """Create a vectorized environment for parallel data collection."""
    logger.info("Creating AsyncVectorEnv with %d parallel envs (base_seed=%d)", n_envs, seed)
    def _thunk(i: int):
        def _init():
            return make_env(env_cfg, seed=seed + i, reward_norm=reward_norm)
        return _init

    vec_env = gym.vector.AsyncVectorEnv([_thunk(i) for i in range(n_envs)])
    logger.info("VectorEnv ready: %d envs", n_envs)
    return vec_env


# Quick sanity check
tmp = make_env(ENV_CONFIG, seed=SEED, reward_norm=True)
print(f"Environment: {ENV_CONFIG['environment']['name']}")
print(f"Observation space: {tmp.observation_space.shape}")
print(f"Action space: {tmp.action_space} (n={tmp.action_space.n})")
N_ACTIONS = tmp.action_space.n
tmp.close()

## 8. Neural Network — Actor-Critic Policy

Nature-DQN style CNN backbone with separate actor (policy) and critic (value) heads.

In [ ]:
class NatureCNN(nn.Module):
    """Nature-DQN convolutional feature extractor.
    Input : (B, C, H, W) float32 in [0, 1]
    Output: (B, feature_dim)
    """

    def __init__(self, in_channels: int, feature_dim: int = 512) -> None:
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=8, stride=4),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU(inplace=True),
        )
        with torch.no_grad():
            dummy = torch.zeros(1, in_channels, 84, 84)
            flat_size = self.conv(dummy).numel()
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_size, feature_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.fc(self.conv(x))


class ActorCriticPolicy(nn.Module):
    """Shared-backbone actor-critic network for discrete PPO."""

    def __init__(self, in_channels: int, n_actions: int, feature_dim: int = 512) -> None:
        super().__init__()
        self.features = NatureCNN(in_channels, feature_dim)
        self.actor_head = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, n_actions),
        )
        self.critic_head = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 1),
        )

    def forward(self, x: Tensor) -> tuple[Tensor, Tensor]:
        feat = self.features(x)
        return self.actor_head(feat), self.critic_head(feat)

    def get_action_and_value(self, x: Tensor, action: Tensor | None = None):
        logits, value = self(x)
        dist = torch.distributions.Categorical(logits=logits)
        if action is None:
            action = dist.sample()
        return action, dist.log_prob(action), dist.entropy(), value.squeeze(-1)

    def get_value(self, x: Tensor) -> Tensor:
        feat = self.features(x)
        return self.critic_head(feat).squeeze(-1)


# Print model summary
in_channels = ENV_CONFIG["preprocessing"]["frame_stack"]
model = ActorCriticPolicy(in_channels, N_ACTIONS)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"ActorCriticPolicy: {total_params:,} total params ({trainable_params:,} trainable)")
print(f"  Input:  ({in_channels}, 84, 84) — {in_channels} stacked grayscale frames")
print(f"  Output: {N_ACTIONS} action logits + 1 state value")
del model

## 9. Utility Functions

In [ ]:
def get_device(preference: str = "auto") -> torch.device:
    """Resolve device: TPU > GPU > CPU."""
    if preference == "auto":
        if _HAS_XLA:
            import torch_xla.core.xla_model as xm
            return xm.xla_device()
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if preference == "tpu" and _HAS_XLA:
        import torch_xla.core.xla_model as xm
        return xm.xla_device()
    return torch.device(preference)


def obs_to_tensor(obs: np.ndarray, device: torch.device) -> Tensor:
    """Convert HWC numpy observation(s) to BCHW float tensor."""
    t = torch.as_tensor(np.ascontiguousarray(obs), dtype=torch.float32)
    if t.ndim == 3:
        t = t.permute(2, 0, 1).unsqueeze(0)
    elif t.ndim == 4:
        t = t.permute(0, 3, 1, 2)
    return t.to(device)


print(f"Utility functions defined. (Device type: {DEVICE_TYPE})")

## 10. PPO Agent

Full PPO implementation with:
- Rollout buffer with GAE (Generalized Advantage Estimation)
- Clipped surrogate objective
- Entropy bonus for exploration
- Mixed-precision training (AMP) on GPU, standard precision on TPU/CPU
- Checkpoint save/load (CPU-compatible for portability)

In [ ]:
class RolloutBuffer:
    """Fixed-length rollout storage for on-policy data collection."""

    def __init__(self, n_steps: int, n_envs: int, obs_shape: tuple) -> None:
        self.n_steps = n_steps
        self.n_envs = n_envs
        self.obs = np.zeros((n_steps, n_envs, *obs_shape), dtype=np.float32)
        self.actions = np.zeros((n_steps, n_envs), dtype=np.int64)
        self.rewards = np.zeros((n_steps, n_envs), dtype=np.float32)
        self.dones = np.zeros((n_steps, n_envs), dtype=np.float32)
        self.log_probs = np.zeros((n_steps, n_envs), dtype=np.float32)
        self.values = np.zeros((n_steps, n_envs), dtype=np.float32)
        self.advantages = np.zeros((n_steps, n_envs), dtype=np.float32)
        self.returns = np.zeros((n_steps, n_envs), dtype=np.float32)
        self._ptr = 0
        logger.debug("RolloutBuffer created: n_steps=%d, n_envs=%d, obs_shape=%s, mem=%.1f MB",
                      n_steps, n_envs, obs_shape,
                      (self.obs.nbytes + self.actions.nbytes + self.rewards.nbytes +
                       self.dones.nbytes + self.log_probs.nbytes + self.values.nbytes +
                       self.advantages.nbytes + self.returns.nbytes) / 1e6)

    def add(self, obs, action, reward, done, log_prob, value) -> None:
        self.obs[self._ptr] = obs
        self.actions[self._ptr] = action
        self.rewards[self._ptr] = reward
        self.dones[self._ptr] = done
        self.log_probs[self._ptr] = log_prob
        self.values[self._ptr] = value
        self._ptr += 1

    def reset(self) -> None:
        self._ptr = 0

    @property
    def full(self) -> bool:
        return self._ptr >= self.n_steps

    def compute_gae(self, last_value, gamma: float, gae_lambda: float) -> None:
        gae = np.zeros(self.n_envs, dtype=np.float32)
        for t in reversed(range(self.n_steps)):
            next_value = last_value if t == self.n_steps - 1 else self.values[t + 1]
            delta = self.rewards[t] + gamma * next_value * (1.0 - self.dones[t]) - self.values[t]
            gae = delta + gamma * gae_lambda * (1.0 - self.dones[t]) * gae
            self.advantages[t] = gae
        self.returns = self.advantages + self.values
        logger.debug("GAE computed: adv mean=%.4f std=%.4f, returns mean=%.4f",
                      self.advantages.mean(), self.advantages.std(), self.returns.mean())

    def flatten(self) -> dict:
        b = self.n_steps * self.n_envs
        return {
            "obs": self.obs.reshape(b, *self.obs.shape[2:]),
            "actions": self.actions.reshape(b),
            "log_probs": self.log_probs.reshape(b),
            "advantages": self.advantages.reshape(b),
            "returns": self.returns.reshape(b),
            "values": self.values.reshape(b),
        }

    def get_batches(self, batch_size: int, rng: np.random.Generator):
        total = self.n_steps * self.n_envs
        indices = rng.permutation(total)
        for start in range(0, total, batch_size):
            yield indices[start : start + batch_size]


class PPOAgent:
    """Production-ready PPO agent for discrete CarRacing.
    Supports GPU (CUDA), TPU (torch_xla), and CPU.
    """

    def __init__(self, train_cfg: dict, env_cfg: dict, n_actions: int) -> None:
        t = train_cfg.get("training", {})
        p = env_cfg.get("preprocessing", {})

        self.gamma = t.get("gamma", 0.99)
        self.gae_lambda = t.get("gae_lambda", 0.95)
        self.clip_range = t.get("clip_range", 0.2)
        self.vf_coef = t.get("vf_coef", 0.5)
        self.ent_coef = t.get("ent_coef", 0.01)
        self.max_grad_norm = t.get("max_grad_norm", 0.5)
        self.n_epochs = t.get("n_epochs", 10)
        self.batch_size = t.get("batch_size", 64)
        self.n_steps = t.get("n_steps", 2048)
        self.normalize_advantage = t.get("normalize_advantage", True)

        # Value function clipping (prevents big jumps in value predictions)
        self.clip_vf = t.get("clip_vf", False)
        self.clip_vf_range = t.get("clip_vf_range", 10.0)

        self.device = get_device(t.get("device", "auto"))
        self.n_actions = n_actions

        # Determine device capabilities
        self._device_type = DEVICE_TYPE  # "cuda", "tpu", or "cpu"
        # AMP only works on CUDA GPUs
        self._use_amp = self._device_type == "cuda"

        in_channels = p.get("frame_stack", 4)
        self.policy = ActorCriticPolicy(in_channels, n_actions).to(self.device)

        logger.info("PPOAgent init: device=%s, n_actions=%d, in_channels=%d", self._device_type, n_actions, in_channels)
        logger.debug("  Hyperparams: gamma=%.3f, gae_lambda=%.3f, clip=%.2f, ent_coef=%.3f, vf_coef=%.2f",
                      self.gamma, self.gae_lambda, self.clip_range, self.ent_coef, self.vf_coef)
        logger.debug("  Training: n_steps=%d, batch_size=%d, n_epochs=%d, max_grad_norm=%.2f",
                      self.n_steps, self.batch_size, self.n_epochs, self.max_grad_norm)
        logger.debug("  Value clipping: %s (range=%.1f)", self.clip_vf, self.clip_vf_range)
        logger.debug("  AMP enabled: %s", self._use_amp)

        # torch.compile for speedup — only on CUDA (not supported on TPU/CPU)
        if hasattr(torch, "compile") and self._device_type == "cuda":
            try:
                self.policy = torch.compile(self.policy)
                logger.info("torch.compile enabled (CUDA)")
            except Exception as e:
                logger.warning("torch.compile failed: %s", e)

        self.optimizer = torch.optim.Adam(
            self.policy.parameters(), lr=t.get("learning_rate", 3e-4), eps=1e-5
        )
        logger.debug("  Optimizer: Adam (lr=%.1e, eps=1e-5)", t.get("learning_rate", 3e-4))

        # GradScaler only for CUDA AMP
        self._scaler = torch.amp.GradScaler("cuda", enabled=self._use_amp)

        resize = tuple(p.get("resize", [84, 84]))
        obs_shape = (*resize, in_channels)
        n_envs = 1
        self.buffer = RolloutBuffer(self.n_steps, n_envs, obs_shape)
        self.rng = np.random.default_rng(t.get("seed", 42))
        self._update_count = 0

    def set_n_envs(self, n: int) -> None:
        obs_shape = self.buffer.obs.shape[2:]
        self.buffer = RolloutBuffer(self.n_steps, n, obs_shape)

    def predict(self, obs: np.ndarray, *, deterministic: bool = False) -> int:
        with torch.no_grad():
            obs_t = obs_to_tensor(obs, self.device)
            logits, _ = self.policy(obs_t)
            if deterministic:
                return int(logits.argmax(dim=-1).item())
            dist = torch.distributions.Categorical(logits=logits)
            return int(dist.sample().item())

    def predict_batch(self, obs: np.ndarray):
        with torch.no_grad():
            obs_t = obs_to_tensor(obs, self.device)
            actions, log_probs, _, values = self.policy.get_action_and_value(obs_t)
        return actions.cpu().numpy(), log_probs.cpu().numpy(), values.cpu().numpy()

    def get_value(self, obs: np.ndarray) -> np.ndarray:
        with torch.no_grad():
            obs_t = obs_to_tensor(obs, self.device)
            return self.policy.get_value(obs_t).cpu().numpy()

    def _compute_vf_loss(self, new_val: Tensor, returns: Tensor, old_values: Tensor) -> Tensor:
        """Compute value loss, optionally with clipping."""
        if self.clip_vf:
            # Clipped value loss (like PPO's clipped policy loss)
            vf_unclipped = (new_val - returns) ** 2
            v_clipped = old_values + torch.clamp(
                new_val - old_values, -self.clip_vf_range, self.clip_vf_range
            )
            vf_clipped = (v_clipped - returns) ** 2
            return 0.5 * torch.max(vf_unclipped, vf_clipped).mean()
        else:
            return torch.nn.functional.mse_loss(new_val, returns)

    def update(self) -> dict:
        logger.debug("PPO update #%d starting (buffer: %d steps x %d envs = %d samples)",
                      self._update_count + 1, self.buffer.n_steps, self.buffer.n_envs,
                      self.buffer.n_steps * self.buffer.n_envs)

        flat = self.buffer.flatten()
        obs_all = torch.as_tensor(flat["obs"]).float().permute(0, 3, 1, 2).to(self.device)
        actions_all = torch.as_tensor(flat["actions"]).long().to(self.device)
        old_log_probs = torch.as_tensor(flat["log_probs"]).float().to(self.device)
        advantages = torch.as_tensor(flat["advantages"]).float().to(self.device)
        returns = torch.as_tensor(flat["returns"]).float().to(self.device)
        old_values = torch.as_tensor(flat["values"]).float().to(self.device)

        logger.debug("  Rollout stats: reward_mean=%.4f, values_mean=%.4f, returns_mean=%.4f, dones=%.1f%%",
                      self.buffer.rewards.mean(), flat["values"].mean(), flat["returns"].mean(),
                      100.0 * self.buffer.dones.mean())

        if self.normalize_advantage and advantages.numel() > 1:
            advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        stats = {"loss": 0.0, "pg_loss": 0.0, "vf_loss": 0.0, "entropy": 0.0,
                 "clip_frac": 0.0, "approx_kl": 0.0, "grad_norm": 0.0}
        n_updates = 0

        for epoch in range(self.n_epochs):
            epoch_stats = {"loss": 0.0, "pg_loss": 0.0, "vf_loss": 0.0, "entropy": 0.0,
                           "clip_frac": 0.0, "approx_kl": 0.0, "grad_norm": 0.0}
            epoch_batches = 0

            for idx in self.buffer.get_batches(self.batch_size, self.rng):
                b_idx = torch.as_tensor(idx).long().to(self.device)

                if self._use_amp:
                    with torch.amp.autocast("cuda", enabled=True):
                        _, new_lp, entropy, new_val = self.policy.get_action_and_value(
                            obs_all[b_idx], actions_all[b_idx]
                        )
                        ratio = torch.exp(new_lp - old_log_probs[b_idx])
                        pg1 = ratio * advantages[b_idx]
                        pg2 = torch.clamp(ratio, 1 - self.clip_range, 1 + self.clip_range) * advantages[b_idx]
                        pg_loss = -torch.min(pg1, pg2).mean()
                        vf_loss = self._compute_vf_loss(new_val, returns[b_idx], old_values[b_idx])
                        ent_loss = -entropy.mean()
                        loss = pg_loss + self.vf_coef * vf_loss + self.ent_coef * ent_loss

                    self.optimizer.zero_grad()
                    self._scaler.scale(loss).backward()
                    self._scaler.unscale_(self.optimizer)
                    grad_norm = torch.nn.utils.clip_grad_norm_(self.policy.parameters(), self.max_grad_norm).item()
                    self._scaler.step(self.optimizer)
                    self._scaler.update()
                else:
                    _, new_lp, entropy, new_val = self.policy.get_action_and_value(
                        obs_all[b_idx], actions_all[b_idx]
                    )
                    ratio = torch.exp(new_lp - old_log_probs[b_idx])
                    pg1 = ratio * advantages[b_idx]
                    pg2 = torch.clamp(ratio, 1 - self.clip_range, 1 + self.clip_range) * advantages[b_idx]
                    pg_loss = -torch.min(pg1, pg2).mean()
                    vf_loss = self._compute_vf_loss(new_val, returns[b_idx], old_values[b_idx])
                    ent_loss = -entropy.mean()
                    loss = pg_loss + self.vf_coef * vf_loss + self.ent_coef * ent_loss

                    self.optimizer.zero_grad()
                    loss.backward()
                    grad_norm = torch.nn.utils.clip_grad_norm_(self.policy.parameters(), self.max_grad_norm).item()
                    if self._device_type == "tpu" and _HAS_XLA:
                        import torch_xla.core.xla_model as xm
                        xm.optimizer_step(self.optimizer)
                    else:
                        self.optimizer.step()

                with torch.no_grad():
                    clip_frac = ((ratio - 1.0).abs() > self.clip_range).float().mean().item()
                    approx_kl = ((ratio - 1.0) - (ratio.log())).mean().item()

                stats["loss"] += loss.item()
                stats["pg_loss"] += pg_loss.item()
                stats["vf_loss"] += vf_loss.item()
                stats["entropy"] += entropy.mean().item()
                stats["clip_frac"] += clip_frac
                stats["approx_kl"] += approx_kl
                stats["grad_norm"] += grad_norm

                epoch_stats["loss"] += loss.item()
                epoch_stats["pg_loss"] += pg_loss.item()
                epoch_stats["vf_loss"] += vf_loss.item()
                epoch_stats["entropy"] += entropy.mean().item()
                epoch_stats["clip_frac"] += clip_frac
                epoch_stats["approx_kl"] += approx_kl
                epoch_stats["grad_norm"] += grad_norm
                epoch_batches += 1
                n_updates += 1

            if epoch_batches > 0:
                logger.debug("  Epoch %d/%d: loss=%.4f pg=%.4f vf=%.4f ent=%.3f clip=%.2f%% kl=%.4f gnorm=%.3f",
                             epoch + 1, self.n_epochs,
                             epoch_stats["loss"] / epoch_batches,
                             epoch_stats["pg_loss"] / epoch_batches,
                             epoch_stats["vf_loss"] / epoch_batches,
                             epoch_stats["entropy"] / epoch_batches,
                             100 * epoch_stats["clip_frac"] / epoch_batches,
                             epoch_stats["approx_kl"] / epoch_batches,
                             epoch_stats["grad_norm"] / epoch_batches)

        self._update_count += 1
        self.buffer.reset()
        result = {k: v / max(n_updates, 1) for k, v in stats.items()}

        logger.info("PPO update #%d done: loss=%.4f pg=%.4f vf=%.4f ent=%.3f clip=%.1f%% kl=%.4f gnorm=%.3f",
                     self._update_count, result["loss"], result["pg_loss"], result["vf_loss"],
                     result["entropy"], 100 * result["clip_frac"], result["approx_kl"], result["grad_norm"])
        return result

    def save(self, path: str) -> None:
        """Save checkpoint. Always saves with CPU tensors for portability."""
        path = pathlib.Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        cpu_policy_sd = {k: v.cpu() for k, v in self.policy.state_dict().items()}
        cpu_optim_sd = self.optimizer.state_dict()
        torch.save(
            {
                "policy": cpu_policy_sd,
                "optimizer": cpu_optim_sd,
                "update_count": self._update_count,
            },
            path,
        )
        logger.info("Saved checkpoint → %s (update #%d)", path, self._update_count)

    def load(self, path: str) -> None:
        data = torch.load(path, map_location="cpu", weights_only=True)
        self.policy.load_state_dict(data["policy"])
        self.policy.to(self.device)
        self.optimizer.load_state_dict(data["optimizer"])
        self._update_count = data.get("update_count", 0)
        logger.info("Loaded checkpoint ← %s (updates=%d)", path, self._update_count)


print(f"PPOAgent defined. (Device: {DEVICE_TYPE})")

## 11. Evaluation Function

In [ ]:
def evaluate_agent(
    agent: PPOAgent,
    env_cfg: dict,
    *,
    n_episodes: int = 5,
    deterministic: bool = True,
    seed: int = 0,
) -> float:
    """Run n_episodes and return the mean total reward (raw, unnormalized)."""
    logger.info("Evaluation starting: %d episodes (deterministic=%s, seed=%d)", n_episodes, deterministic, seed)
    # Disable reward normalization for evaluation — we want the true score
    env = make_env(env_cfg, seed=seed, reward_norm=False)
    rewards = []

    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed + ep)
        total_reward = 0.0
        steps = 0
        done = False
        while not done:
            action = agent.predict(obs, deterministic=deterministic)
            obs, reward, terminated, truncated, _ = env.step(action)
            total_reward += float(reward)
            steps += 1
            done = terminated or truncated
        rewards.append(total_reward)
        logger.debug("  Eval ep %d/%d: reward=%.1f, steps=%d", ep + 1, n_episodes, total_reward, steps)

    env.close()
    mean = float(np.mean(rewards))
    std = float(np.std(rewards))
    logger.info("Evaluation done: mean=%.1f ± %.1f (min=%.1f, max=%.1f)",
                mean, std, min(rewards), max(rewards))
    return mean


print("Evaluation function defined.")

## 12. Training Loop

Vectorized training with periodic evaluation, checkpointing, and metric tracking.

In [ ]:
def find_latest_checkpoint(save_dir: pathlib.Path) -> tuple[str | None, int]:
    """Find the latest checkpoint in save_dir and return (path, global_step).
    Returns (None, 0) if no checkpoint exists.
    """
    if not save_dir.exists():
        return None, 0
    checkpoints = sorted(save_dir.glob("checkpoint_*.pt"))
    if not checkpoints:
        return None, 0
    latest = checkpoints[-1]
    # Extract step number from filename like checkpoint_81920.pt
    try:
        step = int(latest.stem.split("_")[1])
    except (IndexError, ValueError):
        step = 0
    return str(latest), step


def train(
    train_cfg: dict,
    env_cfg: dict,
    n_envs: int = 4,
) -> tuple[PPOAgent, dict]:
    """Full PPO training loop with vectorized environments.
    
    Supports RESUME from the latest checkpoint saved on Google Drive.
    Saves a checkpoint after EVERY rollout so no progress is lost on disconnect.
    
    Returns the trained agent and training history dict.
    """
    t = train_cfg.get("training", {})
    ckpt = train_cfg.get("checkpoints", {})
    ev = train_cfg.get("evaluation", {})

    total_timesteps = t.get("total_timesteps", 2_000_000)
    n_steps = t.get("n_steps", 2048)
    seed = t.get("seed", 42)
    eval_freq = ev.get("eval_freq", 50_000)
    n_eval_episodes = ev.get("n_eval_episodes", 5)
    reward_norm = t.get("reward_normalization", True)

    save_dir = pathlib.Path(ckpt.get("save_dir", DRIVE_SAVE_DIR))
    best_dir = pathlib.Path(ckpt.get("best_model_dir", DRIVE_BEST_DIR))
    save_dir.mkdir(parents=True, exist_ok=True)
    best_dir.mkdir(parents=True, exist_ok=True)

    # Create agent
    tmp = make_env(env_cfg, seed=seed, reward_norm=False)
    n_actions = tmp.action_space.n
    tmp.close()

    agent = PPOAgent(train_cfg, env_cfg, n_actions)

    n_rollouts = total_timesteps // (n_steps * n_envs)
    steps_per_rollout = n_steps * n_envs

    # ---- RESUME from latest checkpoint if available ----
    resume_path, resume_step = find_latest_checkpoint(save_dir)
    start_rollout = 1
    global_step = 0
    best_reward = -float("inf")

    if resume_path is not None:
        agent.load(resume_path)
        global_step = resume_step
        start_rollout = (resume_step // steps_per_rollout) + 1
        # Check if a best model exists and load its reward
        best_model_path = best_dir / "best_model.pt"
        if best_model_path.exists():
            best_ckpt = torch.load(str(best_model_path), map_location="cpu", weights_only=True)
            best_reward = best_ckpt.get("best_reward", -float("inf"))
        logger.info("=" * 60)
        logger.info("RESUMING training from %s", resume_path)
        logger.info("  Resumed at global_step=%s, rollout=%d/%d", f"{global_step:,}", start_rollout, n_rollouts)
        logger.info("  Best reward so far: %.1f", best_reward)
        logger.info("=" * 60)
    else:
        logger.info("No checkpoint found — starting training from scratch.")

    logger.info("=" * 60)
    logger.info("PPO Training — %s timesteps, %d envs", f"{total_timesteps:,}", n_envs)
    logger.info("  Device: %s", agent.device)
    logger.info("  Actions: %d", n_actions)
    logger.info("  Rollouts: %d (%d steps each = %d samples/rollout)", n_rollouts, n_steps, steps_per_rollout)
    logger.info("  Eval every %s steps", f"{eval_freq:,}")
    logger.info("  Checkpoints saved to Google Drive: %s", save_dir)
    logger.info("  Reward normalization: %s", reward_norm)
    logger.info("=" * 60)

    # Create vectorized environment (with reward normalization for training)
    vec_env = make_vec_env(env_cfg, n_envs=n_envs, seed=seed, reward_norm=reward_norm)
    agent.set_n_envs(n_envs)

    obs, _ = vec_env.reset(seed=seed)
    start = time.time()

    # History for plotting
    history = {
        "steps": [], "loss": [], "pg_loss": [], "vf_loss": [],
        "entropy": [], "eval_steps": [], "eval_reward": [], "time": [],
    }

    if start_rollout > n_rollouts:
        logger.info("Training already completed (all %d rollouts done). Nothing to do.", n_rollouts)
        vec_env.close()
        return agent, history

    for rollout in trange(start_rollout, n_rollouts + 1, desc="PPO Training"):
        rollout_start = time.time()

        # ---- Collect rollout ----
        agent.buffer.reset()
        rollout_rewards = []
        rollout_dones = 0
        for step_i in range(n_steps):
            actions, log_probs, values = agent.predict_batch(obs)
            next_obs, rewards, terminated, truncated, infos = vec_env.step(actions)
            dones = np.logical_or(terminated, truncated).astype(np.float32)
            agent.buffer.add(obs, actions, rewards, dones, log_probs, values)
            obs = next_obs
            global_step += n_envs
            rollout_rewards.append(rewards.mean())
            rollout_dones += dones.sum()

        rollout_collect_time = time.time() - rollout_start
        mean_rollout_reward = float(np.mean(rollout_rewards))
        logger.debug("Rollout %d/%d collected: %d steps in %.1fs, mean_reward=%.3f, episodes_done=%d",
                      rollout, n_rollouts, steps_per_rollout, rollout_collect_time,
                      mean_rollout_reward, int(rollout_dones))

        # ---- Compute GAE & Update ----
        last_val = agent.get_value(obs)
        agent.buffer.compute_gae(last_val, agent.gamma, agent.gae_lambda)

        update_start = time.time()
        metrics = agent.update()
        update_time = time.time() - update_start

        # Record metrics
        history["steps"].append(global_step)
        history["loss"].append(metrics["loss"])
        history["pg_loss"].append(metrics["pg_loss"])
        history["vf_loss"].append(metrics["vf_loss"])
        history["entropy"].append(metrics["entropy"])
        history["time"].append(time.time() - start)

        elapsed = time.time() - start
        fps = global_step / elapsed

        logger.debug("Rollout %d/%d update done in %.1fs | step=%s | FPS=%.0f | "
                      "loss=%.4f pg=%.4f vf=%.4f ent=%.3f",
                      rollout, n_rollouts, update_time, f"{global_step:,}", fps,
                      metrics["loss"], metrics["pg_loss"], metrics["vf_loss"], metrics["entropy"])

        # GPU memory logging
        if DEVICE_TYPE == "cuda" and rollout % 10 == 0:
            alloc = torch.cuda.memory_allocated() / 1e9
            reserved = torch.cuda.memory_reserved() / 1e9
            logger.debug("GPU memory: allocated=%.2f GB, reserved=%.2f GB", alloc, reserved)

        # ---- Save checkpoint EVERY rollout to Google Drive (survive disconnects) ----
        agent.save(str(save_dir / f"checkpoint_{global_step}.pt"))

        # ---- Periodic evaluation ----
        if global_step % eval_freq < (n_steps * n_envs):
            mean_r = evaluate_agent(
                agent, env_cfg,
                n_episodes=n_eval_episodes,
                deterministic=True,
                seed=seed + 10_000,
            )
            history["eval_steps"].append(global_step)
            history["eval_reward"].append(mean_r)

            logger.info(
                "Step %8s | eval=%7.1f | loss=%.4f | entropy=%.3f | "
                "clip=%.1f%% | kl=%.4f | gnorm=%.3f | FPS=%.0f | time=%.1fm",
                f"{global_step:,}", mean_r, metrics["loss"], metrics["entropy"],
                100 * metrics.get("clip_frac", 0), metrics.get("approx_kl", 0),
                metrics.get("grad_norm", 0), fps, elapsed / 60,
            )

            if mean_r > best_reward:
                best_reward = mean_r
                # Save best model with reward info for resume
                best_path = str(best_dir / "best_model.pt")
                path_obj = pathlib.Path(best_path)
                path_obj.parent.mkdir(parents=True, exist_ok=True)
                cpu_policy_sd = {k: v.cpu() for k, v in agent.policy.state_dict().items()}
                torch.save({
                    "policy": cpu_policy_sd,
                    "optimizer": agent.optimizer.state_dict(),
                    "update_count": agent._update_count,
                    "best_reward": best_reward,
                }, path_obj)
                logger.info("★ New best model! reward=%.1f → %s", best_reward, best_path)

    # Final save
    agent.save(str(save_dir / "final_model.pt"))

    elapsed = time.time() - start
    logger.info("=" * 60)
    logger.info("Training complete!")
    logger.info("  Total time:    %.1f minutes", elapsed / 60)
    logger.info("  Best reward:   %.1f", best_reward)
    logger.info("  Total steps:   %s", f"{global_step:,}")
    logger.info("  Avg FPS:       %.0f", global_step / elapsed)
    logger.info("  Checkpoints:   %s", save_dir)
    logger.info("  Best model:    %s", best_dir / "best_model.pt")
    logger.info("=" * 60)

    vec_env.close()
    return agent, history


print("Training function defined (with Google Drive checkpointing + resume support).")

## 13. Start Training

Estimated training time for 500K timesteps (61 rollouts):
- **GPU T4/P100**: ~4-5 hours (bottleneck is env stepping at ~27 FPS)
- **TPU v3-8**: ~5-6 hours
- **CPU**: Not recommended

> **Note**: Most time is spent in environment stepping (~300s/rollout), not gradient updates (~1-2s/rollout). The GPU is mostly idle during data collection — Box2D physics simulation on CPU is the bottleneck. 500K steps is enough for the agent to learn a reasonable driving policy.

In [ ]:
agent, history = train(TRAIN_CONFIG, ENV_CONFIG, n_envs=N_ENVS)

## 14. Training Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("PPO Training Progress", fontsize=16, fontweight="bold")

# Evaluation Reward
ax = axes[0, 0]
if history["eval_steps"]:
    ax.plot(history["eval_steps"], history["eval_reward"], "b-o", markersize=4)
    ax.axhline(y=max(history["eval_reward"]), color="r", linestyle="--", alpha=0.5,
               label=f"Best: {max(history['eval_reward']):.1f}")
    ax.legend()
ax.set_xlabel("Timesteps")
ax.set_ylabel("Mean Reward")
ax.set_title("Evaluation Reward")
ax.grid(True, alpha=0.3)

# Total Loss
ax = axes[0, 1]
ax.plot(history["steps"], history["loss"], alpha=0.3)
# Smoothed
window = min(50, len(history["loss"]) // 4 + 1)
if window > 1:
    smoothed = np.convolve(history["loss"], np.ones(window)/window, mode="valid")
    ax.plot(history["steps"][window-1:], smoothed, "r-", linewidth=2, label="Smoothed")
    ax.legend()
ax.set_xlabel("Timesteps")
ax.set_ylabel("Loss")
ax.set_title("Total Loss")
ax.grid(True, alpha=0.3)

# Policy Loss
ax = axes[1, 0]
ax.plot(history["steps"], history["pg_loss"], alpha=0.3, color="green")
if window > 1:
    smoothed = np.convolve(history["pg_loss"], np.ones(window)/window, mode="valid")
    ax.plot(history["steps"][window-1:], smoothed, "darkgreen", linewidth=2, label="Smoothed")
    ax.legend()
ax.set_xlabel("Timesteps")
ax.set_ylabel("Policy Loss")
ax.set_title("Policy Gradient Loss")
ax.grid(True, alpha=0.3)

# Entropy
ax = axes[1, 1]
ax.plot(history["steps"], history["entropy"], alpha=0.3, color="purple")
if window > 1:
    smoothed = np.convolve(history["entropy"], np.ones(window)/window, mode="valid")
    ax.plot(history["steps"][window-1:], smoothed, "purple", linewidth=2, label="Smoothed")
    ax.legend()
ax.set_xlabel("Timesteps")
ax.set_ylabel("Entropy")
ax.set_title("Policy Entropy")
ax.grid(True, alpha=0.3)

plt.tight_layout()
curves_path = os.path.join(DRIVE_BASE, "training_curves.png")
plt.savefig(curves_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved training_curves.png → {curves_path}")

## 15. Final Evaluation

Run a thorough evaluation of the best model.

In [ ]:
# Load the best model for final evaluation
best_path = os.path.join(DRIVE_BEST_DIR, "best_model.pt")

if os.path.exists(best_path):
    best_agent = PPOAgent(TRAIN_CONFIG, ENV_CONFIG, N_ACTIONS)
    best_agent.load(best_path)
    best_agent.policy.eval()

    print("\nFinal Evaluation (20 episodes, deterministic):")
    print("-" * 50)

    env = make_env(ENV_CONFIG, seed=99999)
    episode_rewards = []
    episode_steps = []

    for ep in range(20):
        obs, _ = env.reset(seed=99999 + ep)
        total_reward = 0.0
        steps = 0
        done = False
        while not done:
            action = best_agent.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, _ = env.step(action)
            total_reward += float(reward)
            steps += 1
            done = terminated or truncated
        episode_rewards.append(total_reward)
        episode_steps.append(steps)

    env.close()

    print(f"  Mean reward:  {np.mean(episode_rewards):.1f} ± {np.std(episode_rewards):.1f}")
    print(f"  Min reward:   {np.min(episode_rewards):.1f}")
    print(f"  Max reward:   {np.max(episode_rewards):.1f}")
    print(f"  Mean steps:   {np.mean(episode_steps):.0f}")
    print(f"  Median reward:{np.median(episode_rewards):.1f}")
else:
    print("No best model found. Training may not have completed.")

## 16. Verify Model Compatibility

Ensure the saved model loads correctly and matches the local project's expected format.

In [ ]:
# Verify the checkpoint format matches what the local project expects
print("Model Compatibility Check")
print("=" * 50)
print(f"Trained on: {DEVICE_TYPE.upper()}")

# Load on CPU to verify portability (this is what your local machine will do)
checkpoint = torch.load(best_path, map_location="cpu", weights_only=True)
print(f"Checkpoint keys: {list(checkpoint.keys())}")
print(f"Update count:    {checkpoint.get('update_count', 'N/A')}")

# Verify all tensors are on CPU (important for TPU-trained models)
policy_sd = checkpoint["policy"]
all_cpu = all(t.device.type == "cpu" for t in policy_sd.values())
print(f"All tensors on CPU: {'✓ Yes' if all_cpu else '✗ No — fix needed!'}")

print(f"\nPolicy layers ({len(policy_sd)} tensors):")
total_params = 0
for name, tensor in policy_sd.items():
    print(f"  {name:40s} {str(list(tensor.shape)):>20s}  ({tensor.numel():,} params)")
    total_params += tensor.numel()
print(f"\nTotal parameters: {total_params:,}")

# Test load into a fresh CPU agent to simulate local loading
test_cfg = dict(TRAIN_CONFIG)
test_cfg["training"] = {**test_cfg["training"], "device": "cpu"}
test_agent = PPOAgent(test_cfg, ENV_CONFIG, N_ACTIONS)
test_agent.load(best_path)
test_agent.policy.eval()

# Test inference on CPU
test_env = make_env(ENV_CONFIG, seed=0)
test_obs, _ = test_env.reset(seed=0)
test_action = test_agent.predict(test_obs, deterministic=True)
test_env.close()

print(f"\n✓ Model loads correctly on CPU")
print(f"✓ Inference works (test action: {test_action})")
print(f"\n{'='*50}")
print(f"The model is compatible with your local project.")
print(f"Trained on {DEVICE_TYPE.upper()}, saved as CPU tensors — works everywhere.")
print(f"\nTo use it locally:")
print(f"  1. Download 'best_model.pt' from Google Drive: Car_Racing/models/ppo_best/")
print(f"  2. Place it at: models/ppo_best/best_model.pt")
print(f"  3. Run: python -m src.main race")

## 17. List Output Files

In [ ]:
import glob

print(f"Output files in Google Drive ({DRIVE_BASE}):")
print("=" * 60)
for f in sorted(glob.glob(os.path.join(DRIVE_BASE, "**", "*"), recursive=True)):
    if os.path.isfile(f):
        size = os.path.getsize(f)
        if size > 1024 * 1024:
            size_str = f"{size / 1024 / 1024:.1f} MB"
        elif size > 1024:
            size_str = f"{size / 1024:.1f} KB"
        else:
            size_str = f"{size} B"
        print(f"  {f}  ({size_str})")

print("\n" + "=" * 60)
print("Your checkpoints are safely stored on Google Drive!")
print(f"  Best model: {DRIVE_BEST_DIR}/best_model.pt")
print("\nTo use locally:")

print("  1. Download 'best_model.pt' from Google Drive: Car_Racing/models/ppo_best/")print("  3. Run: python -m src.main race")
print("  2. Place at: models/ppo_best/best_model.pt")